By default, the [ADBC driver for BigQuery](https://adbc-drivers.org/drivers/bigquery/) authenticates with [Application Default Credentials](https://cloud.google.com/docs/authentication/application-default-credentials). On a development machine, that usually means running `gcloud auth application-default login`, which opens a browser and signs in as you. That works for interactive use, but not for an application that has to run unattended.

This recipe shows how to connect using [workload identity federation](https://cloud.google.com/iam/docs/workload-identity-federation) instead. With this approach:

- You don't need to run `gcloud auth application-default login`, because the application authenticates as a service account.
- You don't need to refresh tokens manually, because the driver runs a small program that you provide whenever it needs a new token.
- You don't need a service account key, which many organizations don't allow.

This is, for example, how we configure connections to BigQuery in [Columnar Gateway](https://columnar.tech/gateway).

The [driver documentation](https://adbc-drivers.org/drivers/bigquery/) lists all the authentication options that the driver supports. This recipe is a detailed companion to it that covers this one option from start to finish.

In this notebook, you will:
1. Create a signing key and configure Google Cloud to trust tokens signed with it.
2. Create a service account that can run BigQuery queries.
3. Write a program that issues tokens, and a credential configuration file that tells the driver to run it.
4. Connect to BigQuery in three ways: with connection options, with a `bigquery://` URI, and with a connection profile.

Requirements:
- Python 3.10 or later
- macOS or Linux
- A Google Cloud project with BigQuery enabled
- The [Google Cloud CLI](https://cloud.google.com/sdk/docs/install) (`gcloud`), for the one-time setup only
- Permission to administer the project, for the one-time setup only: Workload Identity Pool Admin, Service Account Admin, Project IAM Admin, and Service Usage Admin, or Owner

## How It Works

Workload identity federation lets an application exchange a token from an identity provider outside Google Cloud for a short-lived Google Cloud access token. Each time the driver needs a new access token, this happens:

1. The driver reads a **credential configuration file**, which says where tokens come from and which service account to use.
2. The driver runs the **token program** named in that file. The program prints a signed [JSON Web Token](https://datatracker.ietf.org/doc/html/rfc7519) (JWT) that identifies your application.
3. The driver sends the JWT to Google's [Security Token Service](https://cloud.google.com/iam/docs/reference/sts/rest), which verifies the signature and checks the JWT against the rules of your **workload identity provider**.
4. The driver uses the result to obtain an access token for your **service account**.
5. The driver uses that access token to call BigQuery.

All of this happens inside the driver. Your code only tells the driver where the credential configuration file is.

Google calls a credential that comes from running a program an *executable-sourced credential*. The alternative that is often shown in examples is a *file-sourced credential*, where the driver reads a JWT from a file. JWTs expire, so a file-sourced credential requires a separate process to keep the file up to date. An executable-sourced credential doesn't.

In this recipe, your application acts as its own identity provider: the token program signs JWTs with a private key that you generate, and you upload the matching public key to Google Cloud. That makes the recipe self-contained, and it works for an application running anywhere. If you already have an identity provider, see [Use Your Own Identity Provider](#Use-Your-Own-Identity-Provider) at the end.

## Setup

Install the required dependencies:

In [1]:
%pip install -q dbc 'adbc-driver-manager>=1.11.0' pyarrow 'pyjwt[crypto]>=2.8.0'

Note: you may need to restart the kernel to use updated packages.


Install the BigQuery ADBC driver with [dbc](https://columnar.tech/dbc). This recipe was tested with version 1.15.0 of the driver:

In [2]:
!dbc install -q 'bigquery>=1.15.0'

Import the required modules:

In [3]:
import json
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path

from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import rsa
from jwt.algorithms import RSAAlgorithm

## Configure Your Project

The setup steps in this recipe use the Google Cloud CLI. If you haven't already, sign in by running `gcloud auth login` in a terminal.

This is an administrator signing in once to create resources. It is not `gcloud auth application-default login`, and the application that connects to BigQuery doesn't use it. You can run the setup steps on one machine and the application on another, and you can do the setup with Terraform or the Google Cloud console if you prefer.

Set your project ID here:

In [4]:
PROJECT_ID = "my-gcp-project"  # Replace with your Google Cloud project ID

Choose names for the resources that this recipe creates. You can leave these as they are:

In [5]:
POOL_ID = "adbc-bigquery-pool"
PROVIDER_ID = "adbc-bigquery-provider"
SERVICE_ACCOUNT_ID = "adbc-bigquery"
SERVICE_ACCOUNT = f"{SERVICE_ACCOUNT_ID}@{PROJECT_ID}.iam.gserviceaccount.com"

# These identify your application in the JWTs that it issues
ISSUER = "https://adbc-bigquery.invalid"
SUBJECT = "bigquery-app"
KEY_ID = "key-1"

The issuer has to be an `https://` URL, but nothing is ever served from it, because you will upload the public key to Google Cloud directly. The `.invalid` top-level domain is [reserved](https://datatracker.ietf.org/doc/html/rfc2606) and can never be registered. If you choose a different issuer, use a domain that you control.

Workload identity pools are identified by project number, not project ID, so look up the project number:

In [6]:
output = !gcloud projects describe {PROJECT_ID} --format="value(projectNumber)"
PROJECT_NUMBER = output[-1]

POOL = f"projects/{PROJECT_NUMBER}/locations/global/workloadIdentityPools/{POOL_ID}"
PROVIDER = f"{POOL}/providers/{PROVIDER_ID}"
print(PROVIDER)

projects/123456789012/locations/global/workloadIdentityPools/adbc-bigquery-pool/providers/adbc-bigquery-provider


Enable the required APIs. This has no effect on any that are already enabled:

In [7]:
!gcloud services enable --project={PROJECT_ID} \
    bigquery.googleapis.com \
    bigquerystorage.googleapis.com \
    cloudresourcemanager.googleapis.com \
    iam.googleapis.com \
    iamcredentials.googleapis.com \
    sts.googleapis.com

Create a directory for the files that the application needs. Only your user account can read it:

In [8]:
WIF_DIR = Path.home() / ".config" / "adbc-bigquery-wif"
WIF_DIR.mkdir(mode=0o700, parents=True, exist_ok=True)

## Create a Signing Key

Generate an RSA key pair. Save the private key to a file that only your user account can read. The `exist_ok=False` argument makes this step fail instead of overwriting a key that already exists:

In [9]:
private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)

private_key_path = WIF_DIR / "private.pem"
private_key_path.touch(mode=0o600, exist_ok=False)
private_key_path.write_bytes(
    private_key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption(),
    )
);

Save the public key as a [JSON Web Key Set](https://datatracker.ietf.org/doc/html/rfc7517) (JWKS), which is the format that Google Cloud expects. This file contains no secrets:

In [10]:
public_jwk = RSAAlgorithm.to_jwk(private_key.public_key(), as_dict=True)
jwks = {
    "keys": [
        {
            "kty": public_jwk["kty"],
            "n": public_jwk["n"],
            "e": public_jwk["e"],
            "kid": KEY_ID,
            "alg": "RS256",
            "use": "sig",
        }
    ]
}
(WIF_DIR / "jwks.json").write_text(json.dumps(jwks, indent=2) + "\n");

## Create the Workload Identity Pool and Provider

A *workload identity pool* is a container for identities from outside Google Cloud. Create one:

In [11]:
!gcloud iam workload-identity-pools create {POOL_ID} \
    --project={PROJECT_ID} \
    --location=global \
    --display-name="ADBC BigQuery"

Created workload identity pool [adbc-bigquery-pool].


A *provider* in the pool defines which tokens to trust. Create a provider that trusts JWTs from your issuer that are signed with your key:

In [12]:
!gcloud iam workload-identity-pools providers create-oidc {PROVIDER_ID} \
    --project={PROJECT_ID} \
    --location=global \
    --workload-identity-pool={POOL_ID} \
    --issuer-uri={ISSUER} \
    --jwk-json-path={WIF_DIR}/jwks.json \
    --attribute-mapping="google.subject=assertion.sub" \
    --attribute-condition="assertion.sub == '{SUBJECT}'"

Created workload identity pool provider [adbc-bigquery-provider].


- `--jwk-json-path` uploads your public key. Google Cloud uses it to verify signatures, so it doesn't need to fetch anything from the issuer's URL.
- `--attribute-mapping` uses the `sub` (subject) claim of the JWT as the name of the identity in the pool.
- `--attribute-condition` rejects JWTs with any other subject.

Because no audience is specified here, the provider requires the `aud` (audience) claim of each JWT to be the full name of the provider. This prevents a JWT that was issued for some other purpose from being used.

## Create the Service Account and Grant Access

Create the service account that your application will act as:

In [13]:
!gcloud iam service-accounts create {SERVICE_ACCOUNT_ID} \
    --project={PROJECT_ID} \
    --display-name="ADBC BigQuery"

Created service account [adbc-bigquery].
Service account email: adbc-bigquery@my-gcp-project.iam.gserviceaccount.com


Grant it the roles that it needs to run queries with ADBC:
- `roles/bigquery.jobUser` allows it to run queries in the project.
- `roles/bigquery.readSessionUser` allows it to read results with the [BigQuery Storage Read API](https://cloud.google.com/bigquery/docs/reference/storage), which the driver uses to fetch results as Arrow data. `roles/bigquery.jobUser` doesn't include this permission. Without it, queries fail with the error `Arrow reader requires roles/bigquery.readSessionUser`.

In [14]:
!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member="serviceAccount:{SERVICE_ACCOUNT}" \
    --role=roles/bigquery.jobUser \
    --condition=None \
    --format=none

!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member="serviceAccount:{SERVICE_ACCOUNT}" \
    --role=roles/bigquery.readSessionUser \
    --condition=None \
    --format=none

Updated IAM policy for project [my-gcp-project].
Updated IAM policy for project [my-gcp-project].


This recipe queries a public dataset, so it doesn't need access to any data of your own. To query your own tables, also grant `roles/bigquery.dataViewer`, preferably on [only the datasets](https://cloud.google.com/bigquery/docs/control-access-to-resources-iam) that the application reads.

Finally, allow your application to act as the service account. The long `principal://` value refers to the identity in your pool whose subject is `bigquery-app`:

In [15]:
!gcloud iam service-accounts add-iam-policy-binding {SERVICE_ACCOUNT} \
    --project={PROJECT_ID} \
    --member="principal://iam.googleapis.com/{POOL}/subject/{SUBJECT}" \
    --role=roles/iam.workloadIdentityUser \
    --condition=None \
    --format=none

Updated IAM policy for serviceAccount [adbc-bigquery@my-gcp-project.iam.gserviceaccount.com].


## Write the Token Program

This is the program that the driver runs when it needs a new token. It signs a JWT and prints it in the [JSON format that Google's authentication libraries expect](https://cloud.google.com/iam/docs/workload-identity-federation-with-other-providers#executable-sourced-credentials):

In [16]:
%%writefile {WIF_DIR}/mint_token.py
import json
import os
import sys
import time
from pathlib import Path

import jwt

# These must match the workload identity provider
ISSUER = "https://adbc-bigquery.invalid"
SUBJECT = "bigquery-app"
KEY_ID = "key-1"

LIFETIME_SECONDS = 3600


def mint_token() -> dict:
    # The driver sets this variable to the name of the workload identity provider
    audience = "https:" + os.environ["GOOGLE_EXTERNAL_ACCOUNT_AUDIENCE"]
    private_key = Path(__file__).with_name("private.pem").read_bytes()
    now = int(time.time())
    expires = now + LIFETIME_SECONDS

    token = jwt.encode(
        {"iss": ISSUER, "sub": SUBJECT, "aud": audience, "iat": now, "exp": expires},
        private_key,
        algorithm="RS256",
        headers={"kid": KEY_ID},
    )
    return {
        "version": 1,
        "success": True,
        "token_type": "urn:ietf:params:oauth:token-type:jwt",
        "id_token": token,
        "expiration_time": expires,
    }


if __name__ == "__main__":
    try:
        print(json.dumps(mint_token()))
    except Exception as e:
        print(json.dumps({"version": 1, "success": False, "code": "1", "message": repr(e)}))
        sys.exit(1)

Writing /Users/me/.config/adbc-bigquery-wif/mint_token.py


The driver passes the audience to the program in the `GOOGLE_EXTERNAL_ACCOUNT_AUDIENCE` environment variable, so the program doesn't need to know your project number or the name of your pool. JWTs can be valid for up to 24 hours. This program makes them valid for one hour, because the driver can always run the program again.

The driver runs the program several times when a connection is first used. After that, it reuses the access token until the token is close to expiring, so later queries on the same connection don't have to wait for the program. If your application opens several connections, the program might run more than once at the same time, so it shouldn't modify any shared state.

Run the program once to check that it works. Only the start of the token is shown, because a complete token is a usable credential until it expires:

In [17]:
result = subprocess.run(
    [sys.executable, WIF_DIR / "mint_token.py"],
    env={**os.environ, "GOOGLE_EXTERNAL_ACCOUNT_AUDIENCE": f"//iam.googleapis.com/{PROVIDER}"},
    capture_output=True,
    text=True,
)
response = json.loads(result.stdout)
if response["success"]:
    response["id_token"] = response["id_token"][:20] + "..."
response

{'version': 1,
 'success': True,
 'token_type': 'urn:ietf:params:oauth:token-type:jwt',
 'id_token': 'eyJhbGciOiJSUzI1NiIs...',
 'expiration_time': 1795551868}

## Create the Credential Configuration File

The credential configuration file tells the driver which provider to use, which service account to act as, and which program to run:

In [18]:
command = f"{sys.executable} {WIF_DIR}/mint_token.py"
assert len(command.split()) == 2, "The paths in the command must not contain spaces"

!gcloud iam workload-identity-pools create-cred-config {PROVIDER} \
    --service-account={SERVICE_ACCOUNT} \
    --executable-command="{command}" \
    --executable-timeout-millis=30000 \
    --output-file={WIF_DIR}/credentials.json

Created credential configuration file [/Users/me/.config/adbc-bigquery-wif/credentials.json].


The command must use an absolute path to the program, and no path in it can contain spaces, because the driver splits the command wherever there is a space. Here, the command runs the same Python interpreter that is running this notebook, because that is where the `pyjwt` package is installed.

Here is the resulting file:

In [19]:
print((WIF_DIR / "credentials.json").read_text())

{
  "universe_domain": "googleapis.com",
  "type": "external_account",
  "audience": "//iam.googleapis.com/projects/123456789012/locations/global/workloadIdentityPools/adbc-bigquery-pool/providers/adbc-bigquery-provider",
  "subject_token_type": "urn:ietf:params:oauth:token-type:jwt",
  "token_url": "https://sts.googleapis.com/v1/token",
  "credential_source": {
    "executable": {
      "command": "/Users/me/columnar-cookbook/.pixi/envs/default/bin/python /Users/me/.config/adbc-bigquery-wif/mint_token.py",
      "timeout_millis": 30000
    }
  },
  "service_account_impersonation_url": "https://iamcredentials.googleapis.com/v1/projects/-/serviceAccounts/adbc-bigquery@my-gcp-project.iam.gserviceaccount.com:generateAccessToken"
}


This file contains no secrets, but it does contain a command that the driver will run, so don't let untrusted users modify it. The same goes for `mint_token.py`. Here is what each file is for:

| File | Used by | Secret? |
| --- | --- | --- |
| `private.pem` | `mint_token.py`, to sign JWTs | **Yes** |
| `jwks.json` | You, once, to create the provider | No |
| `mint_token.py` | The driver, to obtain JWTs | No |
| `credentials.json` | The driver, to find everything else | No |

Anyone who has `private.pem` can act as your service account, so protect it as you would protect a password. To replace it, see [Rotate the Signing Key](#Rotate-the-Signing-Key) at the end.

## Allow the Driver to Run the Program

As a safety measure, Google's authentication libraries refuse to run programs unless the environment variable `GOOGLE_EXTERNAL_ACCOUNT_ALLOW_EXECUTABLES` is set to `1`. Without it, connections fail with the error `executables need to be explicitly allowed`.

**Set this variable before your application starts**, for example in your shell, your service definition, or your container image. Don't rely on setting it from inside the application. When we tested this on macOS, setting it with `os.environ` in a Python process that was already running had no effect on a driver loaded later in that same process.

For that reason, the examples below are complete scripts that each run in a new Python process, as they would in a real deployment. Processes started from this notebook inherit variables that are set here, so set the variable now, along with one that tells the scripts which project to use:

In [20]:
os.environ["GOOGLE_EXTERNAL_ACCOUNT_ALLOW_EXECUTABLES"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

## Connect with Connection Options

This is a full example in Python. It connects by specifying `driver="bigquery"` and passing the connection options in `db_kwargs`. Then it runs two queries. The first one shows which identity the connection is using, and the second one fetches some data as an Arrow table:

In [21]:
%%writefile {WIF_DIR}/connect_with_options.py
import os
from pathlib import Path

from adbc_driver_manager import dbapi

project_id = os.environ["GOOGLE_CLOUD_PROJECT"]
credentials = Path.home() / ".config" / "adbc-bigquery-wif" / "credentials.json"

with (
    dbapi.connect(
        driver="bigquery",
        db_kwargs={
            "bigquery.project_id": project_id,
            "bigquery.dataset_id": "my_dataset",
            "bigquery.auth_type": "json_credential_file",
            "bigquery.auth.credentials": str(credentials),
        },
        autocommit=True,
    ) as connection,
    connection.cursor() as cursor,
):
    cursor.execute("SELECT SESSION_USER() AS identity")
    print("Connected as", cursor.fetch_arrow_table()["identity"][0])

    cursor.execute("""
        SELECT word, word_count
        FROM `bigquery-public-data.samples.shakespeare`
        WHERE corpus = 'hamlet'
        ORDER BY word_count DESC, word
        LIMIT 5
    """)
    table = cursor.fetch_arrow_table()

print(table)

Writing /Users/me/.config/adbc-bigquery-wif/connect_with_options.py


Changes to permissions in Google Cloud typically take about two minutes to take effect, and sometimes seven minutes or longer. Because you just granted the permissions, retry the script until it succeeds:

In [22]:
for _ in range(40):
    result = subprocess.run(
        [sys.executable, WIF_DIR / "connect_with_options.py"], capture_output=True, text=True
    )
    if result.returncode == 0:
        break
    time.sleep(15)

print(result.stdout or result.stderr)

Connected as adbc-bigquery@my-gcp-project.iam.gserviceaccount.com
pyarrow.Table
word: string
word_count: int64
----
word: [["the","and","to","of","I"]]
word_count: [[995,706,635,630,546]]



The connection is using the service account, not your own Google account. Here is what each option does:

| Option | Purpose |
| --- | --- |
| `bigquery.project_id` | The project to run queries in. |
| `bigquery.dataset_id` | The dataset in your project where BigQuery looks for tables when a query doesn't specify a dataset. Version 1.15.0 of the driver requires this. The dataset doesn't have to exist if, as here, your queries use fully qualified table names. |
| `bigquery.auth_type` | `json_credential_file` tells the driver to read credentials from a JSON file. |
| `bigquery.auth.credentials` | The path to the credential configuration file. |

The driver detects that the file describes workload identity federation, so you don't need to set `bigquery.auth.credentials_type`. If you do set it, use `external_account`.

Don't set any of the driver's `bigquery.impersonate.*` options. The credential configuration file already specifies the service account.

## Connect with a URI

You can specify the same options in a `bigquery://` URI. The URI has this form:

```
bigquery:///PROJECT_ID?OAuthType=1&AuthCredentials=PATH&DatasetId=DATASET_ID
```

`OAuthType=1` corresponds to `json_credential_file`. The driver documentation describes this value as "Service Account JSON File", but it works for credential configuration files too. Special characters in the URI must be percent-encoded, so encode the path:

In [23]:
%%writefile {WIF_DIR}/connect_with_uri.py
import os
from pathlib import Path
from urllib.parse import quote

from adbc_driver_manager import dbapi

project_id = os.environ["GOOGLE_CLOUD_PROJECT"]
credentials = Path.home() / ".config" / "adbc-bigquery-wif" / "credentials.json"

uri = (
    f"bigquery:///{project_id}"
    "?OAuthType=1"
    f"&AuthCredentials={quote(str(credentials), safe='')}"
    "&DatasetId=my_dataset"
)

with (
    dbapi.connect(uri, autocommit=True) as connection,
    connection.cursor() as cursor,
):
    cursor.execute("SELECT SESSION_USER() AS identity")
    print("Connected as", cursor.fetch_arrow_table()["identity"][0])

Writing /Users/me/.config/adbc-bigquery-wif/connect_with_uri.py


In [24]:
!{sys.executable} {WIF_DIR}/connect_with_uri.py

Connected as adbc-bigquery@my-gcp-project.iam.gserviceaccount.com


You don't need to specify the driver separately. The driver manager gets it from the `bigquery://` at the start of the URI.

## Connect with a Connection Profile

An ADBC [connection profile](https://arrow.apache.org/adbc/main/format/connection_profiles.html) is a TOML file that stores connection options, so that your code doesn't have to. To learn more, see [Use ADBC Connection Profiles in Python](https://cookbook.columnar.tech/use-adbc-connection-profiles-in-python).

Create a profile named `bigquery_wif` in the directory where the driver manager looks for profiles:

In [25]:
if platform.system() == "Darwin":
    profile_dir = Path.home() / "Library" / "Application Support" / "ADBC" / "Profiles"
else:
    xdg = os.environ.get("XDG_CONFIG_HOME", Path.home() / ".config")
    profile_dir = Path(xdg) / "adbc" / "profiles"

profile_dir.mkdir(parents=True, exist_ok=True)
profile_path = profile_dir / "bigquery_wif.toml"

profile_path.write_text(f"""\
profile_version = 1
driver = "bigquery"

[Options]
"bigquery.project_id" = "{PROJECT_ID}"
"bigquery.dataset_id" = "my_dataset"
"bigquery.auth_type" = "json_credential_file"
"bigquery.auth.credentials" = "{WIF_DIR}/credentials.json"
""")
print(profile_path.read_text())

profile_version = 1
driver = "bigquery"

[Options]
"bigquery.project_id" = "my-gcp-project"
"bigquery.dataset_id" = "my_dataset"
"bigquery.auth_type" = "json_credential_file"
"bigquery.auth.credentials" = "/Users/me/.config/adbc-bigquery-wif/credentials.json"



Connect by specifying the name of the profile. The code contains no other connection details:

In [26]:
%%writefile {WIF_DIR}/connect_with_profile.py
from adbc_driver_manager import dbapi

with (
    dbapi.connect("profile://bigquery_wif", autocommit=True) as connection,
    connection.cursor() as cursor,
):
    cursor.execute("SELECT SESSION_USER() AS identity")
    print("Connected as", cursor.fetch_arrow_table()["identity"][0])

Writing /Users/me/.config/adbc-bigquery-wif/connect_with_profile.py


In [27]:
!{sys.executable} {WIF_DIR}/connect_with_profile.py

Connected as adbc-bigquery@my-gcp-project.iam.gserviceaccount.com


Connection profiles and URIs aren't specific to Python. You can use the same profile and the same URI from any language that has an ADBC driver manager.

A profile can't set `GOOGLE_EXTERNAL_ACCOUNT_ALLOW_EXECUTABLES` for you. It is an environment variable, not a connection option, so it must be set in the environment whichever way you connect.

## Deploy Your Application

To run your application on another machine, provide:

1. The BigQuery ADBC driver.
2. `private.pem`, `mint_token.py`, and `credentials.json`. Store `private.pem` in a secrets manager if you have one.
3. A Python interpreter with `pyjwt[crypto]` installed, to run `mint_token.py`. The token program can be written in any language, so if your application isn't written in Python, you might prefer to rewrite it.
4. The environment variable `GOOGLE_EXTERNAL_ACCOUNT_ALLOW_EXECUTABLES=1`.

Update the paths in the `command` field of `credentials.json` to match that machine. You don't need `gcloud` there, and you don't need to sign in.

### Set It Up Once with Columnar Gateway

Repeating this for every environment that connects to BigQuery is tedious and error-prone:

- Every laptop, server, container image, and CI job needs all four of the things listed above, with paths in `credentials.json` that are correct for that environment.
- If they share a signing key, every copy of `private.pem` is one more secret to protect, and [rotating the key](#Rotate-the-Signing-Key) means replacing all of them.
- If they don't share a key, you have more keys to create, upload, and keep track of.
- When something breaks, you have to work out which of these pieces is wrong in which environment.

With [Columnar Gateway](https://columnar.tech/gateway), you set this up once, in Gateway. Your applications and agents then connect to BigQuery through Gateway from anywhere, so the signing key, the token program, and the credential configuration file stay in one place. The same goes for the other databases that you connect to.

To learn more, see [columnar.tech/gateway](https://columnar.tech/gateway).

## Rotate the Signing Key

A provider can trust up to eight keys at once, so you can replace the signing key without interrupting your application:

1. Generate a new key pair, and give it a new key ID, such as `key-2`.
2. Add the new public key to `jwks.json` alongside the old one, and upload the file using `gcloud iam workload-identity-pools providers update-oidc` with the `--jwk-json-path` option.
3. Replace `private.pem` with the new private key, and change `KEY_ID` in `mint_token.py`.
4. Remove the old public key from `jwks.json`, and upload the file again. After this, the old private key is useless.

## Use Your Own Identity Provider

In this recipe, the signing key is a long-lived secret. If your application runs somewhere that already gives it an identity, you can avoid having one. Change where the token comes from, and leave the ADBC connection code as it is:

- **AWS or Azure**: Create a provider for that platform and generate the credential configuration file with the `--aws` or `--azure` option. The driver obtains tokens from the platform, so you don't need a token program. See [Workload identity federation with AWS or Azure](https://cloud.google.com/iam/docs/workload-identity-federation-with-other-clouds).
- **Kubernetes outside Google Cloud**: Kubernetes can write a service account token to a file and keep it current, so use a file-sourced credential. See [Workload identity federation with Kubernetes](https://cloud.google.com/iam/docs/workload-identity-federation-with-kubernetes).
- **Google Kubernetes Engine or Compute Engine**: You don't need a credential configuration file. Attach a service account to the workload and use the driver's default authentication.
- **Another OpenID Connect or SAML identity provider**: Create a provider that trusts it, and replace `mint_token.py` with a program that obtains a token from it and prints the token in the same JSON format.

## Cleanup

To remove the resources that this recipe created in Google Cloud, revoke the roles, and then delete the service account and the workload identity pool. Deleting the pool also deletes the provider:

In [28]:
!gcloud projects remove-iam-policy-binding {PROJECT_ID} \
    --member="serviceAccount:{SERVICE_ACCOUNT}" \
    --role=roles/bigquery.jobUser \
    --condition=None \
    --format=none

!gcloud projects remove-iam-policy-binding {PROJECT_ID} \
    --member="serviceAccount:{SERVICE_ACCOUNT}" \
    --role=roles/bigquery.readSessionUser \
    --condition=None \
    --format=none

!gcloud iam service-accounts delete {SERVICE_ACCOUNT} --project={PROJECT_ID} --quiet

!gcloud iam workload-identity-pools delete {POOL_ID} \
    --project={PROJECT_ID} \
    --location=global \
    --quiet

Updated IAM policy for project [my-gcp-project].
Updated IAM policy for project [my-gcp-project].
deleted service account [adbc-bigquery@my-gcp-project.iam.gserviceaccount.com]
Deleted workload identity pool [adbc-bigquery-pool].


Google Cloud keeps deleted pools for 30 days, and you can't reuse the name of a deleted pool during that time. To run this recipe again within 30 days, change `POOL_ID`.

Remove the local files, including the private key:

In [29]:
profile_path.unlink(missing_ok=True)
shutil.rmtree(WIF_DIR)